# Prerequisites

## Create Catalog and schema

In [0]:
%sql
use catalog ecommerce;
create schema if not exists bronze;

## Create DFs

In [0]:
inventory_df = spark.read.csv(
    path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/initial/inventory/inventory_2026-08-01.csv",
    header=True,
    inferSchema=True,
)

products_df = spark.read.csv(
    path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/initial/products/products_2026-08-01.csv",
    header=True,
    inferSchema=True,
)


In [0]:
products_df.display()

## Create dim tables

In [0]:
inventory_df.write.saveAsTable("bronze.inventory", mode="APPEND")
products_df.write.saveAsTable("bronze.products", mode="APPEND")

In [0]:
# from pyspark.sql.types import *
# from pyspark.sql.functions import *

# CLICKSTREAM_SCHEMA = StructType(
#     [
#         StructField("event_id", IntegerType(), True),
#         StructField("customer_id", DoubleType(), True),
#         StructField("session_id", StringType(), True),
#         StructField("event_time", TimestampType(), True),
#         StructField("event_type", StringType(), True),
#         StructField("product_id", IntegerType(), True),
#         StructField("page", StringType(), True),
#         StructField("device_type", StringType(), True),
#     ]
# )

# input_df = spark.readStream.load(
#     format="CSV",
#     path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/initial/clickstream/",
#     schema=CLICKSTREAM_SCHEMA,
#     header=True,
# )

# input_df.writeStream.option(
#     "checkpointLocation",
#     "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/clickstream/checkpoint",
# ).trigger(availableNow=True).outputMode("APPEND").toTable("ecommerce.bronze.clickstream")

## Read Tables

In [0]:
spark.read.table("bronze.clickstream").limit(5).display()
spark.read.table("bronze.products").limit(5).display()
spark.read.table("bronze.inventory").limit(5).display()

# Check dups

In [0]:
%sql
SELECT
    event_id,
    COUNT(*) AS cnt
FROM ecommerce.bronze.clickstream
GROUP BY event_id
HAVING COUNT(*) > 1
limit 3;

# Clickstream using AutoLoaders

In [0]:
%sql
-- drop table if exists ecommerce.bronze.clickstream;

In [0]:
soruce_configurations = {
    "cloudFiles.format":"json",
    "timestampFormat":"yyyy-MM-dd",
    "cloudFiles.schemaLocation":"/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/autoloaders/clickstream/schema/",
    "cloudFiles.inferColumnTypes":True
}
 
destination_configurations = {
    "checkpointLocation": "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/autoloaders/clickstream/checkpoint/",
    "mergeSchema": True,
}
source_df = (
    spark.readStream.format("cloudFiles")
    .options(**soruce_configurations)
    .load("/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/adls/initial/clickstream/")
)

source_df.writeStream.format("DELTA").options(**destination_configurations).outputMode(
    "APPEND"
).trigger(availableNow=True).toTable("ecommerce.bronze.clickstream")

In [0]:
display(spark.read.table("ecommerce.bronze.clickstream").count())
spark.read.table("ecommerce.bronze.clickstream").display()